# Lab 6 – Linear Regression
**Dataset:** Medical Insurance Cost (`insurance.csv`)  
**Target:** `charges` — predict individual medical insurance cost (USD)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('insurance.csv')
df.head()

## 2. Explore the Data

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(df['age'],      df['charges'], alpha=0.4, s=15, color='steelblue')
axes[0].set_xlabel('Age');      axes[0].set_ylabel('Charges'); axes[0].set_title('Age vs Charges')
axes[1].scatter(df['bmi'],      df['charges'], alpha=0.4, s=15, color='darkorange')
axes[1].set_xlabel('BMI');      axes[1].set_ylabel('Charges'); axes[1].set_title('BMI vs Charges')
axes[2].scatter(df['children'], df['charges'], alpha=0.4, s=15, color='green')
axes[2].set_xlabel('Children'); axes[2].set_ylabel('Charges'); axes[2].set_title('Children vs Charges')
plt.tight_layout(); plt.show()

## 3. Data Cleaning & Feature Engineering

In [ ]:
print('Missing values:'); print(df.isnull().sum())

df_model = df.copy()

# Encode binary categoricals
df_model['sex_enc']    = (df_model['sex']    == 'male').astype(int)
df_model['smoker_enc'] = (df_model['smoker'] == 'yes').astype(int)

# One-hot encode region
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)

# Drop original string columns
df_model.drop(columns=['sex','smoker'], inplace=True)

print('\nFinal feature columns:')
print(df_model.columns.tolist())

## 4. Train / Test Split

In [ ]:
X = df_model.drop('charges', axis=1)
y = df_model['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'Training set : {X_train.shape}')
print(f'Test set     : {X_test.shape}')

## 5. Train Linear Regression Model

In [ ]:
lm = LinearRegression()
lm.fit(X_train, y_train)

coeff_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lm.coef_})
coeff_df = coeff_df.sort_values('Coefficient', ascending=False)
print('Model Coefficients:')
print(coeff_df.to_string(index=False))
print(f'\nIntercept: {lm.intercept_:.2f}')

## 6. Evaluate the Model

In [ ]:
predictions = lm.predict(X_test)

MAE  = mean_absolute_error(y_test, predictions)
MSE  = mean_squared_error(y_test, predictions)
RMSE = np.sqrt(MSE)

print(f'MAE  (Mean Absolute Error)      : ${MAE:,.2f}')
print(f'MSE  (Mean Squared Error)       : ${MSE:,.2f}')
print(f'RMSE (Root Mean Squared Error)  : ${RMSE:,.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted
axes[0].scatter(y_test, predictions, alpha=0.5, s=20, color='steelblue', edgecolors='none')
lims = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect Prediction')
axes[0].set_xlabel('Actual Charges'); axes[0].set_ylabel('Predicted Charges')
axes[0].set_title('Actual vs Predicted'); axes[0].legend()

# Residuals
residuals = y_test - predictions
sns.histplot(residuals, bins=35, kde=True, color='darkorange', ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Residual'); axes[1].set_title('Residuals Distribution')

plt.tight_layout(); plt.show()
print(f'Residuals skewness: {pd.Series(residuals).skew():.3f}')

**Interpretation:**  
`smoker_enc` has by far the largest positive coefficient, confirming that smoking is the dominant driver of insurance cost. `age` and `bmi` also have significant positive coefficients. The residuals are roughly normal but slightly right-skewed due to the smoker high-cost cluster, suggesting a log-transformed target or a non-linear model would further reduce error.